# Member 3 — BOM cost, pricing and revenue
This notebook follows the three-, six- and nine-month forecasts for RM6 and RM AirSeT. The preceding pipeline supplies the material-price forecasts and historical outcomes. Member 3 reconciles the BOM, estimates cost per unit, compares product-level forecast errors and tests how prices and volumes affect revenue.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "member3.py").is_file())
out = root / "outputs" / "member3"
rebuild = False
if rebuild or not (out / "product_holdout_scores.csv").exists():
    subprocess.run([sys.executable, str(root / "src" / "member3.py")], cwd=root, check=True)

## BOM audit
Both workbooks use the current product weight as the coverage denominator. The legacy RM6 carbon-steel line retains the team's imputed weight in a separate category. Lines missing usable prices retain their mass and an empty cost. The sensitivity tests apply €1, €3 or €6 per kg to unpriced mass, and test a ±25% change in the PET and engineering-plastics anchors.

In [ ]:
coverage = pd.read_csv(out / "bom_coverage.csv")
shares = coverage.pivot_table(index=["product", "bom_version"], columns="coverage_status",
                              values="mass_share", fill_value=0)
display(shares.map("{:.1%}".format))
display(Image(filename=str(out / "figures" / "01_bom_coverage.png"), width=1000))
display(Image(filename=str(out / "figures" / "05_bom_sensitivity.png"), width=1000))

## Cost and forecast validation
The September v1 BOM weights translate the Member 2 material forecasts into a bottom-up cost. Indexed polymers move from the price anchor at each forecast origin. The forecast errors are measured against realized market proxies on the same basket.

The primary comparison uses common origins after December 2024. The Member 2 family strategy and bands are chosen from outcomes available by that date. The final-origin strategy applied across all historical origins is a descriptive comparison and is exported separately. Chronos-2 was also run as a separate benchmark. See README for its MAE and release-date constraints.

The main product P50 is flat cost. Its product interval is estimated from the historical errors of that same flat point. The material stress sums each material's individual tail forecast at once. The joint stress and the empirical product quantile have different interpretations.

In [ ]:
cost = pd.read_csv(out / "product_costs.csv")
scores = pd.read_csv(out / "product_holdout_scores.csv")
cost_fields = ["product", "target_month", "priced_mass_share", "origin_material_cost_eur_per_unit",
               "central_material_cost_eur_per_unit", "planning_p10_eur_per_unit",
               "planning_p50_eur_per_unit", "planning_p90_eur_per_unit", "high_stress_material_cost_eur_per_unit"]
display(cost[cost_fields].round(2))
display(scores.pivot(index=["product", "h"], columns="method", values="mae_eur_per_unit").round(2))
plausibility = pd.read_csv(out / "material_band_plausibility.csv")
display(plausibility.loc[plausibility["p90_above_max_rise"],
                        ["series", "h", "label", "p90_pct", "hist_max_rise"]].round(3))
for name in ["02_cost_drivers", "03_unit_cost_forecast", "04_product_backtest"]:
    display(Image(filename=str(out / "figures" / f"{name}.png"), width=1000))

## Pricing and revenue
The scenarios hold price, pass through 50% or 100% of the central material-cost move, or cover the simultaneous high stress. The pass-through examples use the Member 2 bottom-up change. The formal ETO baseline uses the flat product point. The adopted planning method and the held-out MAE winners are recorded in `pricing_decisions.csv`. The default 2026 synthetic anchors are Schneider for RM6 and Eaton for AirSeT. The break-even volume is the number of units at the new price that matches base-price revenue.

For custom scenarios, run `uv run python src/member3.py --commercial-inputs PATH` with a CSV keyed by `product,target_month`. Optional columns are `units`, `base_price_eur_ex_vat` and `additional_direct_cost_eur_per_unit`. Units describe baseline volume at the base price for that month. The extra direct cost should cover omitted materials and other direct costs. The elasticity scenarios are assumptions that show the implications of different volume responses.

In [ ]:
prices = pd.read_csv(out / "pricing_revenue.csv")
decision = pd.read_csv(out / "pricing_decisions.csv")
last = prices.loc[prices["h"].eq(prices["h"].max()), ["product", "policy", "selling_price_eur_ex_vat",
                  "break_even_volume_change_for_revenue", "high_stress_cost_change_net_of_price_eur_per_unit"]].copy()
last["break_even_volume_change_for_revenue"] = last["break_even_volume_change_for_revenue"].map("{:+.2%}".format)
display(last.round(2))
display(decision[["product", "h", "lowest_mae_method", "planning_method", "planning_cost_reference_eur_per_unit"]].round(2))
for name in ["06_price_and_cost_risk", "07_revenue_and_elasticity"]:
    display(Image(filename=str(out / "figures" / f"{name}.png"), width=1000))

## How to defend the decision

- **Why MAE?** If we price one euro of over- and under-estimation equally, the conditional median minimizes expected absolute error. We compare methods with paired errors on the same origins. The Newey–West intervals allow for overlapping horizons and the multiple-comparison adjustment covers both products, all three horizons and all three alternative methods in the held-out comparison.
- **Why this quote?** Let \(x\) be the uplift, \(H\) the stress cost increase and \(B\) the allowed revenue loss. Under constant price elasticity, \(R/R_0=(1+x/p_0)^{1+\varepsilon}\). The lowest elasticity is the worst case for any uplift. When it is less than −1, the revenue floor implies \(x\leq x_{\max}=p_0[(1-B)^{1/(1+\varepsilon_{\min})}-1]\). The uncovered cost \(\max(0,H-x)\) decreases up to \(H\). Choosing the smallest uplift among minimizers gives \(x^*=\min(H,x_{\max})\). The code enforces the bounds at cent precision.
- **What Member 4 owns.** The frontier is indexed by stress basis, elasticity floor and acceptable revenue decline. Member 4 can calibrate these inputs by segment and compare the remaining cost exposure with purchasing actions, quote terms or a phased rollout. `member4_material_exposure.csv` supplies the material-level quantities and stress contributions.

In [ ]:
tests = pd.read_csv(out / "forecast_decision_tests.csv")
display(tests[["evaluation", "product", "h", "method", "mae_minus_flat_eur_per_unit",
               "ci_95_simultaneous_low", "ci_95_simultaneous_high", "p_bonferroni"]].round(3))
frontier = pd.read_csv(out / "member4_pricing_frontier.csv")
case = frontier.loc[frontier["h"].eq(frontier["h"].max())
                    & frontier["elasticity_floor"].eq(-2)
                    & frontier["maximum_revenue_loss"].eq(0.01)]
display(case[["product", "stress_basis", "optimal_uplift_eur", "uncovered_cost_eur_per_unit",
              "minimum_revenue_loss_budget_for_full_cover"]].round(4))

### Re-run the pricing decision for your segment
Edit the revenue loss limit, elasticity floor, cost stress or price anchor below. Provide the current segment quote in euros to replace the synthetic anchor.

In [ ]:
sys.path.insert(0, str(root / "src"))
from member3_decision_analysis import optimal_quote

product = "RM6_NE_IQI_433520"
horizon = 9
stress = "simultaneous_high"
revenue_loss_limit = 0.01
elasticity_floor = -2.0
segment_base_quote = None

row = frontier.loc[frontier["product"].eq(product) & frontier["h"].eq(horizon)
                   & frontier["stress_basis"].eq(stress)].iloc[0]
base_price = row.base_price_eur_ex_vat if segment_base_quote is None else segment_base_quote
solution = optimal_quote(base_price, row.stress_increase_eur_per_unit, elasticity_floor, revenue_loss_limit)
display(pd.Series(solution).round(4))


## Final ETO policy and team submission

Each row pairs a delivery horizon with a cost scenario. The validity is the longest tested acceptance window within the waiting-risk allowance. The downside buffer combines the greater of the flat-point empirical upper bound and simultaneous material stress with the waiting reserve. Base and downside prices respect the worst-case revenue constraint, and an upside discount requires confirmed cost savings. The indexation safeguard shows both the contractual surcharge needed and the smaller surcharge compatible with the demand constraint. Any unmet protection remains a purchasing or commercial decision.

The paper accepts a normalized product index. The formal CSV uses the same v1 BOM as the offer policy and calibrates its product bounds on the selected flat point model. It supplies one forecast per registered material and canonical product index at each horizon. The coverage register documents lines and categories without a validated price mapping.

In [ ]:
policy = pd.read_csv(out / "eto_quote_policy.csv")
display(policy[["product", "h", "scenario", "downside_cost_basis", "selected_validity_calendar_months", "illustrative_offer_eur_ex_vat",
                "index_trigger_level", "required_non_price_protection_at_downside_eur_per_unit"]].round(2))
display(pd.read_csv(out / "submission_bom_calibration.csv").round(3))
for name in ["01_quote_validity", "02_offer_downside", "03_index_clause", "04_migration_threshold"]:
    display(Image(filename=str(out / "eto_figures" / f"{name}.png"), width=1000))

### Re-evaluate indexation when the monthly basket is published
Use the signed initial adjustment and change the illustrative basket reading below. The outputs are a decision aid for an agreed clause or a pre-award review.

In [ ]:
from member3_eto import OfferAssumptions, indexation_impact

quote_row = policy.loc[policy["product"].eq(product) & policy["h"].eq(horizon)
                       & policy["scenario"].eq("downside")].iloc[0]
observed_index = 130.0
updated = indexation_impact(quote_row.base_price_eur_ex_vat, quote_row.price_adjustment_eur,
                            quote_row.base_bom_cost_eur_per_unit, observed_index, OfferAssumptions())
display(pd.Series(updated))